# 06 — Analyse d'erreurs — le verdict, la ventilation et les erreurs expliquées

**Projet** : Routage de tickets de support par TF-IDF — la référence explicable

## Objectifs pédagogiques

1. Lire le verdict contractuel et le situer face aux références triviales.
1. Ventiler les métriques par style : chiffrer ce que les paraphrases coûtent.
1. Identifier la classe sacrifiée et lire la matrice de confusion.
1. Expliquer les erreurs les plus confiantes terme à terme, et conclure par des recommandations chiffrées.

## 1. Mise en place

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (320 tickets) quand `data/raw` est
# vide : l'exécution reste rapide. Si le corpus de référence a été généré par `make data`, il est
# utilisé tel quel — les valeurs absolues d'un notebook ne sont pas celles de la référence.
NB_DOCUMENTS = 320

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks lisent le corpus du projet et écrivent leurs artefacts dans `outputs/notebooks` :
# ils ne doivent jamais écraser ceux produits par `make train` / `make evaluate`.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PATHS.data_dir,
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
TARGET_COLUMN = str(CONFIG.data.target or "label")
LABELS = [
    "facturation",
    "livraison",
    "produit_defectueux",
    "remboursement",
    "compte_client",
    "autre",
]
STYLES = [
    "canonique",
    "paraphrase",
    "bruite",
]

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.7)")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Classes           : {len(LABELS)} — {', '.join(LABELS)}")
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticTicketGenerator
from src.data.loaders import TextLabelLoader

LOADER = TextLabelLoader(
    PATHS,
    formats=CONFIG.data.formats,
    validation_enabled=CONFIG.data.validation.raw,
    lazy_validation=CONFIG.data.validation.lazy,
)
if LOADER.documents_path.exists():
    DOCUMENTS = LOADER.load_documents()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis data/raw : {len(DOCUMENTS)} tickets, graine {METADATA.get('seed')}")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté pour que les pipelines (train, evaluate) trouvent les mêmes fichiers que le
    # notebook ; `make data` régénère le corpus de référence.
    bundle = SyntheticTicketGenerator.from_config(CONFIG.data, seed=CONFIG.seed).generate()
    DOCUMENTS, METADATA = bundle.documents, bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_metadata(METADATA)
    print(f"data/raw absent : corpus de {len(DOCUMENTS)} tickets généré et persisté")

TRAIN = LOADER.split("train", frame=DOCUMENTS)
VALIDATION = LOADER.split("val", frame=DOCUMENTS)
CALIBRATION = LOADER.split("calibration", frame=DOCUMENTS)
TEST = LOADER.split("test", frame=DOCUMENTS)
# La calibration sert à **régler** (seuil d'automatisation), la validation à **choisir**
# (algorithme, rééquilibrage), et le test à **juger** — une seule fois, à la fin.
print(
    "splits :",
    {
        "train": len(TRAIN),
        "val": len(VALIDATION),
        "calibration": len(CALIBRATION),
        "test": len(TEST),
    },
)

LABEL_COUNTS = DOCUMENTS[TARGET_COLUMN].value_counts()
MAJORITY_LABEL = str(LABEL_COUNTS.index[0])
MINORITY_LABEL = str(LABEL_COUNTS.index[-1])
print(
    f"classe majoritaire : {MAJORITY_LABEL} ({LABEL_COUNTS.iloc[0] / len(DOCUMENTS):.1%}) "
    f"| classe minoritaire : {MINORITY_LABEL} ({LABEL_COUNTS.iloc[-1] / len(DOCUMENTS):.1%})"
)

## 2. Le pipeline d'évaluation

Il recharge les artefacts produits par l'entraînement et mesure le split de **test**, une seule fois. Si le notebook est ouvert sur un projet fraîchement cloné, il entraîne d'abord : un notebook doit rester exécutable de bout en bout.

In [ ]:
from src.pipelines import EvaluationPipeline, TrainPipeline

model_file = NB_PATHS.models_dir / "classifier.joblib"
if not model_file.exists():
    print("artefact absent : entraînement sur le corpus réduit…")
    TrainPipeline(CONFIG, paths=NB_PATHS).run()

evaluation_result = EvaluationPipeline(CONFIG, paths=NB_PATHS).run()
assert evaluation_result.succeeded, evaluation_result.messages
evaluated = evaluation_result.payload
for message in evaluation_result.messages:
    print("  -", message)

## 3. Verdict et niveau à battre

La métrique principale est comparée à son seuil contractuel **et** aux références triviales. Un verdict « conforme » qui ne bat pas la classe majoritaire ne vaudrait rien ; c'est pourquoi les deux lectures sont côte à côte.

In [ ]:
primary = CONFIG.metrics.primary
print(f"verdict : {evaluated.verdict}")
print(
    f"{primary} = {evaluated.metrics[primary]:.4f} "
    f"(seuil {evaluated.threshold}, {evaluated.n_documents} tickets de test)"
)

baselines = pd.DataFrame(evaluated.baselines).T
baselines["marge du modèle"] = (evaluated.metrics[primary] - baselines[primary]).round(4)
display(baselines.round(4))

headline = pd.DataFrame(
    {
        "indicateur": [
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "balanced_accuracy",
            "cohen_kappa",
            "mean_confidence",
            "expected_calibration_error",
        ]
    }
).set_index("indicateur")
headline["valeur"] = [round(float(evaluated.metrics[name]), 4) for name in headline.index]
display(headline)

**Ce qu'il faut retenir**

- Le verdict se lit sur la F1 macro : sur six classes dont une minoritaire, c'est la seule moyenne qui ne récompense pas l'abandon de la classe difficile.
- La marge contre la classe majoritaire et contre le tirage stratifié est l'argument du projet : tout le reste n'est que de l'habillage.
- La confiance moyenne se lit **contre** l'exactitude : si elle est plus basse, le modèle se sous-estime, ce qui est préférable à l'inverse pour un seuil d'automatisation.

## 4. Ventilation par style : ce que les paraphrases coûtent

C'est la lecture qui distingue « lire un mot » de « comprendre une demande ». Si l'écart entre le segment canonique et le segment paraphrase est nul, soit la tâche est trop facile, soit le corpus a été écrit avec les mêmes mots partout.

In [ ]:
segments = evaluated.segments.copy()
display(segments.round(4))

style_rows = segments[segments["segment"].str.startswith("style=")].set_index("segment")
display(style_rows[["n_documents", primary]].round(4))
if {"style=canonique", "style=paraphrase"} <= set(style_rows.index):
    canon = float(style_rows.loc["style=canonique", primary])
    para = float(style_rows.loc["style=paraphrase", primary])
    print(
        f"écart canonique - paraphrase sur {primary} : {canon - para:+.4f} "
        f"({canon:.4f} contre {para:.4f})"
    )
    print("taille des segments :", style_rows["n_documents"].to_dict())
else:
    print("segments de style absents du corpus courant")

# Les chiffres du corpus **complet**, extraits du manifeste : le corpus réduit des notebooks ne
# fait que quelques dizaines de tickets de test, et un segment trop petit peut inverser un écart.
REFERENCE_RESULTS = [
    ("F1 macro sur le split de test (six classes, chacune pesant pareil)", "0,8408"),
    ("Exactitude sur le split de test (plancher trivial 0,2034)", "0,8531"),
    ("F1 macro du segment canonique (93 tickets) / du segment paraphrase (52)", "0,9262 / 0,7352"),
    ("F1 macro du segment bruité (32 tickets : politesse et signature partagées)", "0,7487"),
    ("Référence « règle à mots-clés » publiée avec le corpus", "0,5492"),
    ("Kappa de Cohen / exactitude équilibrée", "0,8217 / 0,8337"),
    ("Écart de calibration (confiance moyenne 0,6168 contre exactitude 0,8531)", "0,2363"),
    ("Latence p50 / p95 d'une prédiction (TF-IDF, 6 classes)", "0,67 ms / 0,78 ms"),
    ("Formulations réservées par classe et par style, jamais vues à l'entraînement", "2 sur 6"),
]
display(
    pd.DataFrame(
        REFERENCE_RESULTS, columns=["mesure de référence (corpus complet)", "valeur"]
    ).set_index("mesure de référence (corpus complet)")
)

In [ ]:
figure, axis = plt.subplots(figsize=(7, 3.5))
style_rows[primary].plot.barh(ax=axis, color="#4c72b0")
axis.axvline(
    float(evaluated.metrics[primary]),
    color="#c44e52",
    linestyle="--",
    label=f"moyenne ({evaluated.metrics[primary]:.3f})",
)
axis.set_xlabel(primary)
axis.set_title("Par style rédactionnel — la moyenne cache l'écart")
axis.legend()
figure.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Sur le corpus complet, le segment canonique est nettement au-dessus du segment paraphrase : le modèle lit un vocabulaire et paie l'absence de ces mots — la limite **assumée** de l'approche lexicale.
- Un écart de signe **inverse** n'est pas une bonne nouvelle : sur un petit segment, quelques tickets suffisent à retourner une moyenne. On lit donc toujours l'écart avec la taille des segments, jamais seul.
- Le segment bruité se situe entre les deux : la politesse et la signature ne portent pas d'information, mais elles diluent le signal — sa F1 se lit avec la longueur du segment.
- L'écart canonique / paraphrase est **le** résultat publié de la famille, et il se lit dans les deux sens : sur le corpus de référence, la variante `with-transformers` (encodeur appris sur le corpus) obtient 0,8285 de F1 macro sur les paraphrases contre 0,7352 pour l'approche lexicale — mais 0,7892 sur le canonique contre 0,9262. Un modèle contextuel ne dépend plus du vocabulaire de sa catégorie ; en revanche, il ne bat pas un modèle lexical sur les tournures les plus standardisées.

## 5. La classe sacrifiée

Une F1 macro est une moyenne : elle cache nécessairement une classe qui souffre. On la cherche dans le tableau par classe, puis dans la matrice de confusion.

In [ ]:
per_class = evaluated.per_class.sort_values("f1").reset_index(drop=True)
display(per_class.round(4))

sacrificed = str(per_class.loc[0, "class"])
print(
    f"classe la plus fragile : {sacrificed} (F1 {per_class.loc[0, 'f1']:.4f}, "
    f"{int(per_class.loc[0, 'support'])} tickets)"
)
print("vers quoi la classe fragile se trompe-t-elle ?")
display(evaluated.confusion.loc[sacrificed].sort_values(ascending=False).to_frame("prédictions"))

In [ ]:
from src.visualization.plots import plot_confusion, plot_per_class

print(
    "figure de confusion :",
    plot_confusion(evaluated.confusion, NB_PATHS.figures_dir / "06_confusion.png").name,
)
print(
    "figure par classe   :",
    plot_per_class(evaluated.per_class, NB_PATHS.figures_dir / "06_f1_per_class.png").name,
)

**Ce qu'il faut retenir**

- La classe la plus fragile est presque toujours la plus rare **et** la plus ambiguë pour l'approche retenue : c'est elle qui décide de la F1 macro.
- La matrice de confusion nomme la paire de classes à travailler : sans elle, on sait qu'on se trompe, pas où.
- Les deux figures sont produites par les mêmes fonctions que le rapport : le notebook ne fabrique pas une visualisation parallèle.

## 6. Les erreurs les plus confiantes, expliquées

Les erreurs les plus confiantes sont celles qui coûtent le plus cher en production : un ticket routé avec une forte confiance vers le mauvais service repart pour un tour. Elles sont archivées par le pipeline, et le modèle linéaire sait dire **quels termes** l'ont emporté.

In [ ]:
from src.models import load_model

reloaded = load_model(NB_PATHS.models_dir / "classifier.joblib", config=node(CONFIG, "model"))
errors = evaluated.top_errors
text_by_id = dict(zip(evaluated.predictions["doc_id"], evaluated.predictions["text"], strict=True))

for position, row in enumerate(errors.head(5).itertuples(), start=1):
    text = text_by_id.get(str(row.doc_id), "")
    print(
        f"--- erreur {position} : attendu « {row.expected_label} », "
        f"prédit « {row.prediction} » (confiance {row.confidence:.3f})"
    )
    print(f"    {text[:200]}")
    terms = reloaded.explain([text], k=4)[0]
    print("    termes décisifs :", ", ".join(f"{term} ({weight:+.2f})" for term, weight in terms))
    print()

print(f"erreurs archivées : {len(errors)} (les plus confiantes du split de test)")

**Ce qu'il faut retenir**

- Savoir expliquer une erreur est ce qui permet de la corriger : ajouter une tournure au corpus, revoir une règle de service, ou accepter la limite et envoyer le ticket en relecture.
- Un modèle linéaire s'explique par ses termes : c'est un argument de déploiement dans une équipe support, pas un détail technique.
- Les erreurs confiantes sont minoritaires : le modèle se trompe surtout là où il hésite — ce qui est précisément ce que le seuil d'automatisation exploite.

## 7. Le rapport d'évaluation

Le même pipeline écrit un rapport Markdown et ses tables CSV : verdict, métriques, références, ventilation par classe et par segment, erreurs, figures, lecture honnête et reproductibilité. On en relit ici la section « Lecture honnête ».

In [ ]:
report_path = NB_PATHS.reports_dir / "evaluation_report.md"
report_text = report_path.read_text(encoding="utf-8")
start = report_text.find("## 9.")
print(report_text[start : start + 1100] if start >= 0 else report_text[-1100:])

## 8. Recommandations

Ce que ce notebook permet d'écrire noir sur blanc pour la suite du projet.

In [ ]:
recommendations = [
    f"Router automatiquement au-delà du seuil arbitré au notebook 04 : {CONFIG.metrics.primary} "
    "reste la métrique contractuelle, mesurée sur l'ensemble du test.",
    f"Surveiller le rappel de la classe « {MINORITY_LABEL} » à chaque réentraînement : c'est la "
    "première classe qui souffre quand la distribution des tickets glisse.",
    "Rejouer la ventilation par style après chaque changement de corpus : un écart "
    "canonique / paraphrase qui se réduit est le signe d'un corpus qui s'appauvrit.",
    "Reporter l'écart entre cette variante et `with-transformers` à chaque réentraînement : il "
    "mesure si le corpus a grossi au point de justifier un encodeur, au lieu de le supposer.",
    "Relire les erreurs les plus confiantes avant toute mise en production : c'est là que se "
    "cachent les tournures ambiguës, pas dans les erreurs de faible confiance.",
]
for position, line in enumerate(recommendations, start=1):
    print(f"{position}. {line}")

**Ce qu'il faut retenir**

- Le notebook et le rapport racontent la même histoire : verdict, ventilation, erreurs, recommandations — avec des chiffres, jamais des adjectifs.
- La conclusion honnête de ce projet est un couple : « excellent sur les tickets qui emploient le vocabulaire de leur classe, mesuré sur les paraphrases ».
- Toute la suite du dépôt (variante `transformers`, fusion des deux stacks) se juge sur le même protocole : corpus identique, splits identiques, test lu une seule fois.